# Silver Layer: Region Table Cleaning & Validation

Cleanses `skyledger_airlines.bronze.region` into `skyledger_airlines.silver.region`.
Applies primary key validation.
Region is the top-level geographic parent — no foreign keys to validate.
Invalid records are quarantined in `silver.region_quarantine` with comma-separated rejection reasons.
Includes Bronze-Silver-Quarantine reconciliation and a data-quality report.

**Relationships:** `region` → `nation` (region is the parent of nation in the geographic hierarchy).

In [0]:
# ── Step 1: Read Bronze, Cast Types, Drop Metadata Columns ──
from pyspark.sql.functions import col, when, lit, count as _count, concat_ws
from pyspark.sql.window import Window

bronze_region = spark.table("skyledger_airlines.bronze.region")

# Drop metadata columns: _ingestion_timestamp, _source_file, _pipeline_run_id
# Cast key column from string to INT for proper numeric handling
region_typed = bronze_region.select(
    col("r_regionkey").cast("int").alias("r_regionkey"),
    col("r_name").cast("string").alias("r_name"),
    col("r_comment").cast("string").alias("r_comment"),
)

bronze_count = region_typed.count()
print(f"Bronze region rows read: {bronze_count}")
display(region_typed)

In [0]:
# ── Step 2: Primary Key Validation (§3.1) ──
# Rule: r_regionkey must be NOT NULL and UNIQUE across all rows.
# No dirty records are expected in this reference table;
# any violation is a critical data fault.
# Failure codes: NULL_PK, DUPLICATE_PK

region_flagged = region_typed.withColumn(
    "is_null_pk",
    when(col("r_regionkey").isNull(), lit(True)).otherwise(lit(False)),
)

# Window-based duplicate detection
pk_window = Window.partitionBy("r_regionkey")
region_flagged = region_flagged.withColumn(
    "_pk_count", _count("*").over(pk_window)
).withColumn(
    "is_duplicate_pk",
    when((col("r_regionkey").isNotNull()) & (col("_pk_count") > 1), lit(True)).otherwise(lit(False)),
)

null_pk_count = region_flagged.filter(col("is_null_pk")).count()
duplicate_pk_count = region_flagged.filter(col("is_duplicate_pk")).count()

print(f"NULL_PK failures:      {null_pk_count}")
print(f"DUPLICATE_PK failures: {duplicate_pk_count}")

In [0]:
# ── Step 3: Build Rejection Reasons, Split Clean vs Quarantine, Write Tables ──

# Build comma-separated rejection_reason from all flag columns
region_flagged = region_flagged.withColumn(
    "rejection_reason",
    concat_ws(
        ", ",
        when(col("is_null_pk"), lit("NULL_PK")),
        when(col("is_duplicate_pk"), lit("DUPLICATE_PK")),
    ),
)

# Clean records: passed every check (rejection_reason is empty string)
silver_region = region_flagged.filter(col("rejection_reason") == "").select(
    "r_regionkey", "r_name", "r_comment"
)

# Quarantined records: failed one or more checks
quarantine_region = region_flagged.filter(col("rejection_reason") != "").select(
    "r_regionkey", "r_name", "r_comment", "rejection_reason"
)

silver_count = silver_region.count()
quarantine_count = quarantine_region.count()

print(f"Silver rows:     {silver_count}")
print(f"Quarantine rows: {quarantine_count}")

# Write Silver table (overwrite for idempotency — same result on every run)
(silver_region.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skyledger_airlines.silver.region"))

# Write Quarantine table (overwrite for idempotency)
(quarantine_region.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skyledger_airlines.silver.region_quarantine"))

print("✅ Tables written: silver.region, silver.region_quarantine")

In [0]:
# ── Step 4: Bronze-Silver-Quarantine Reconciliation 
# Bronze row count must equal Silver + Quarantine for the region table.

bronze_total = spark.table("skyledger_airlines.bronze.region").count()
silver_total = spark.table("skyledger_airlines.silver.region").count()
quarantine_total = spark.table("skyledger_airlines.silver.region_quarantine").count()

print(f"Bronze row count:      {bronze_total}")
print(f"Silver row count:      {silver_total}")
print(f"Quarantine row count:  {quarantine_total}")
print(f"Silver + Quarantine:   {silver_total + quarantine_total}")

assert bronze_total == silver_total + quarantine_total, (
    f"RECONCILIATION FAILED: Bronze ({bronze_total}) != "
    f"Silver ({silver_total}) + Quarantine ({quarantine_total})"
)

print("✅ Reconciliation PASSED: Bronze = Silver + Quarantine")

In [0]:
# ── Step 5: Data Quality Report ──
# Documents every check executed, even when zero rows fail.
# Region is the top-level geographic parent — no FK checks apply.

dq_report = spark.createDataFrame(
    [
        ("region", "NULL_PK", "r_regionkey must not be null", null_pk_count, "Quarantine"),
        ("region", "DUPLICATE_PK", "r_regionkey must be unique across all rows", duplicate_pk_count, "Quarantine"),
    ],
    ["table", "rule_code", "check_description", "failing_rows", "action_taken"],
)

display(dq_report)

# Show quarantined records if any exist
if quarantine_count > 0:
    print(f"\n── Quarantined Records ({quarantine_count} rows) ──")
    display(spark.table("skyledger_airlines.silver.region_quarantine"))
else:
    print("\nNo quarantined records — all 5 region rows passed every check.")

# Show final Silver table
print(f"\n── Silver Region Table ({silver_count} rows) ──")
display(spark.table("skyledger_airlines.silver.region"))